# Sensitivity Sweep (full Indonesia)

Standalone notebook for the LSEQM+DL parameter sensitivity analysis (thesis Section 4.8), run over the **full Indonesia domain** rather than the Bali subdomain. It re-runs the correction and metric pipeline while varying, one at a time:

- `DL_BLEND_ALPHA` (CNN blend weight, default 0.70)
- `GPD_THRESHOLD_PERCENTILE` (default 80)
- `DENSITY_SATURATION_COUNT` (default 2)

For each setting it pools the per-pixel CPC-validated metrics over land pixels and the chosen dekads and saves `sensitivity_sweep_indonesia.csv` to the output directory.

**Safe for long runs.** The sweep is resumable and can run one setting at a time (`RUNS_PER_CALL = 1`): each execution of the last cell runs a single setting, checkpoints the CSV, and returns. Re-run that cell for the next setting. A Colab disconnect costs at most the one run in flight.

**This is expensive.** Full Indonesia is ~19,393 land pixels versus Bali's 126, so a single setting is many hours. Colab Pro (or better) is recommended. Do a cheap smoke test first (see the last cell).

## 1 Connect Google Drive (Colab only)

In [ ]:
from google.colab import drive
import os

if os.path.exists('/content/drive'):
    try:
        drive.flush_and_unmount()
        print('Successfully unmounted')
    except Exception:
        print('Unmount failed, the drive might not be mounted or busy')

drive.mount('/content/drive')

## 2 Install packages (only if needed)

In [ ]:
# In Google Colab almost all packages are already available, except netCDF4
!pip install netCDF4

## 3 Setup environment and load configuration

In [ ]:
import sys, os, importlib, logging
logging.basicConfig(level=logging.INFO)

# Windows DLL fix for local conda (no-op on Colab)
if sys.platform == 'win32':
    _p = os.environ.get('CONDA_PREFIX') or sys.prefix
    for _d in [os.path.join(_p,'Library','bin'), os.path.join(_p,'Library','lib'),
               os.path.join(_p,'Library','mingw-w64','bin'), os.path.join(_p,'bin'), _p]:
        if os.path.isdir(_d):
            try: os.add_dll_directory(_d)
            except OSError: pass
            if _d not in os.environ.get('PATH',''):
                os.environ['PATH'] = _d + os.pathsep + os.environ.get('PATH','')

# Project root: Colab Drive path by default; change for local Jupyter.
ROOT = '/content/drive/MyDrive/hybrid-bias-correction'
# ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))   # local
assert os.path.isfile(os.path.join(ROOT,'src','config.py')), f'Not found: {ROOT}'

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
for m in [k for k in sys.modules if k.startswith('src')]:
    del sys.modules[m]

import src.config as _cfg
importlib.reload(_cfg)
CONFIG_FILE = 'config.yml'   # full Indonesia (not config_bali.yml)
_cfg.initialize_config(os.path.join(ROOT, CONFIG_FILE))

from src import config
print('Config loaded:')
print('  output_dir :', config.output_dir)
print('  IMERGL     :', config.imergl_file)
print('  CPC        :', config.cpc_file)
print('  alpha      :', config.DL_BLEND_ALPHA)
print('  gpd pctl   :', config.GPD_THRESHOLD_PERCENTILE)
print('  saturation :', config.DENSITY_SATURATION_COUNT)

## 4 Load datasets (IMERG-L, CPC, native CPC) + land-sea mask

In [ ]:
import xarray as xr
from src import config
from src.utility import apply_land_sea_mask

_engine = config.NETCDF_ENGINE
imerg_ds = xr.open_dataset(config.imergl_file, decode_times=True, engine=_engine)
cpc_ds   = xr.open_dataset(config.cpc_file,   decode_times=True, engine=_engine)

if hasattr(config,'cpc_native_file') and os.path.isfile(config.cpc_native_file):
    cpc_ds_native = xr.open_dataset(config.cpc_native_file, decode_times=True, engine=_engine)
else:
    cpc_ds_native = None
    print('WARNING: native 0.5deg CPC not found; block artefacts may appear')

imerg_ds[config.IMERG_PRECIP_VAR] = apply_land_sea_mask(
    imerg_ds[config.IMERG_PRECIP_VAR], config.mask_file)
cpc_ds[config.CPC_PRECIP_VAR] = apply_land_sea_mask(
    cpc_ds[config.CPC_PRECIP_VAR], config.mask_file)
print('Datasets loaded and land-sea mask applied.')

## 5 Align CPC to the IMERG grid

In [ ]:
from src.utility import load_mask, reindex_and_align_with_monotonicity

land_sea_mask = load_mask(config.mask_file)
cpc_ds_aligned, _ = reindex_and_align_with_monotonicity(imerg_ds, cpc_ds, land_sea_mask)
print('Alignment complete: imerg_ds, cpc_ds_aligned, cpc_ds_native are ready.')

## 6 Run the sensitivity sweep (resumable)

The cell below is preset for a safe long run: `RUNS_PER_CALL = 1` (one setting per execution) and the full 36 dekads.

**Recommended order:**

1. **Smoke test.** Uncomment the 4-dekad `DEKADS` subset line, run the cell once, and confirm you see `checkpointed 1/15 -> .../sensitivity_sweep_indonesia.csv`.
2. **Full run.** Re-comment the subset so `DEKADS` is the full 36, then keep re-running the cell. It skips settings already in the CSV, so it resumes exactly where it stopped. Set `RUNS_PER_CALL = None` to run all remaining settings unattended in one execution.

Progress is printed each time (`k/15 done`) and the CSV is written after every setting.

In [ ]:
# Resumable sensitivity sweep over the full Indonesia domain.
# Reuses imerg_ds, cpc_ds_aligned, cpc_ds_native defined above.
# Edit RUN_TAG / RUNS_PER_CALL / DEKADS in the config block below.

import importlib
import inspect
import glob
import os
import numpy as np
import pandas as pd
import xarray as xr

import src.config as config
import src.utility as _util
import src.station_density as _sd
from src.bias_correction import run_correction_pipeline
from src.metrics import run_metrics_pipeline

# ------------------------------------------------------------------
# Sweep configuration - edit here
# ------------------------------------------------------------------
RUN_TAG = "indonesia"     # names the checkpoint CSV; change per region
RUNS_PER_CALL = 1         # 1 = one heavy setting per cell execution (safest on
                          # Colab Pro); None = run all remaining in one go.

# All 36 dekadal windows (months 1-12, dekads 1-3). This is the full run.
DEKADS = [(m, d) for m in range(1, 13) for d in (1, 2, 3)]
# DEKADS = [(1, 2), (4, 2), (7, 2), (10, 2)]   # quick subset: wet/shoulder/dry

SWEEPS = {
    "alpha": {"param": "DL_BLEND_ALPHA",           "values": [0.5, 0.6, 0.7, 0.8, 0.9], "default": 0.70},
    "gpd":   {"param": "GPD_THRESHOLD_PERCENTILE", "values": [70, 75, 80, 85, 90],       "default": 80},
    "csat":  {"param": "DENSITY_SATURATION_COUNT", "values": [1, 2, 3, 4, 5],            "default": 2},
}

POOL_VARS = ["pearson_correlation", "relative_bias", "stdev_ratio", "rmse", "nse"]
METHOD = "lseqmdl"

_SRC_MODULES = [
    "src.config", "src.distribution_fitting", "src.deep_learning",
    "src.bias_correction", "src.station_density", "src.metrics",
]


def set_param(name, value):
    """Broadcast a config override into every module that binds it.

    Two traps handled here:
    (1) Modules that do `from .config import X` hold their own binding, so we
        patch the name in every module that has it (not just config).
    (2) apply_deeplearning_model() captures blend_alpha as a DEFAULT ARGUMENT
        frozen at import time, and bias_correction calls it WITHOUT passing
        blend_alpha. Patching the module global is therefore not enough for
        DL_BLEND_ALPHA - we must rewrite the function's __defaults__ too, or the
        alpha sweep is a silent no-op.
    """
    n = 0
    for modname in _SRC_MODULES:
        mod = importlib.import_module(modname)
        if hasattr(mod, name):
            setattr(mod, name, value)
            n += 1

    extra = ""
    if name == "DL_BLEND_ALPHA":
        dl = importlib.import_module("src.deep_learning")
        f = dl.apply_deeplearning_model
        sig = inspect.signature(f)
        new_defaults = tuple(
            value if pn == "blend_alpha" else p.default
            for pn, p in sig.parameters.items()
            if p.default is not inspect.Parameter.empty
        )
        f.__defaults__ = new_defaults
        eff = inspect.signature(f).parameters["blend_alpha"].default
        extra = f"; apply_deeplearning_model blend_alpha default -> {eff}"

    print(f"    set {name} = {value}  (patched in {n} module(s)){extra}")


# Metric NetCDFs use the start-day-of-dekad token in their filename:
#   dekad index 1 -> "dekad01", 2 -> "dekad11", 3 -> "dekad21".
DEKAD_TOKEN = {1: "01", 2: "11", 3: "21"}

_PURGE_SUBDIRS = [
    "corrected_ls", "corrected_lseqm", "corrected_lseqmdl",
    "metrics_ls", "metrics_lseqm", "metrics_lseqmdl",
    "quality_ls", "quality_lseqm", "quality_lseqmdl",
]


def reset_for_new_setting():
    """Clear caches and force overwrite before a new parameter setting."""
    _util.reset_user_choice()                  # clear cached file-skip decision
    config.EXISTING_FILE_ACTION = "overwrite"  # belt-and-suspenders
    _sd._confidence_cache.clear()              # drop cached confidence mask
    cmf = getattr(config, "CONFIDENCE_MASK_FILE", None)
    if cmf and os.path.isfile(cmf):
        try:
            os.remove(cmf)
        except OSError:
            pass


def purge_dekad_outputs(m, d):
    """Delete corrected/metrics/quality/model files for one dekad so they regenerate."""
    tok = DEKAD_TOKEN[d]
    pat = f"*month{m:02d}_dekad{tok}*"
    for sub in _PURGE_SUBDIRS:
        for f in glob.glob(f"{config.output_dir}/{sub}/{pat}"):
            try:
                os.remove(f)
            except OSError:
                pass
    mdl = (f"{config.output_dir}/trained_models/"
           f"bias_correction_model_month{m:02d}_dekad{tok}.keras")
    if os.path.isfile(mdl):
        try:
            os.remove(mdl)
        except OSError:
            pass


def pool_metrics(dekads):
    """Pool land-pixel medians from the CPC daily-timeseries metric file.

    Anchors strictly on metricsts_cpc: the imergl/imergf files compare the
    corrected product against the satellite it was derived from and carry
    near-unity correlation that would corrupt the pool.
    """
    rows = {v: [] for v in POOL_VARS}
    for (m, d) in dekads:
        tok = DEKAD_TOKEN[d]
        patt = (f"{config.output_dir}/metrics_{METHOD}/"
                f"*metricsts_cpc_imergl_{METHOD}_month{m:02d}_dekad{tok}*.nc4")
        files = sorted(glob.glob(patt))
        if not files:
            print(f"    WARNING: no metricsts_cpc file for month{m:02d} dekad{tok}")
        for f in files:
            ds = xr.open_dataset(f, decode_timedelta=False)
            for v in POOL_VARS:
                if v in ds:
                    a = ds[v].values.astype(float).ravel()
                    rows[v].extend(a[~np.isnan(a)].tolist())
            ds.close()
    return {v: (float(np.median(rows[v])) if rows[v] else np.nan) for v in POOL_VARS}


def run_setting(dekads):
    """Run correction + metrics for the dekads at the current config, return pooled medians."""
    reset_for_new_setting()
    for (m, d) in dekads:
        purge_dekad_outputs(m, d)
        run_correction_pipeline(imerg_ds, cpc_ds_aligned, m, d,      # noqa: F821
                                cpc_native_ds=cpc_ds_native)         # noqa: F821
        run_metrics_pipeline(m, d, mode="timeseries")
    return pool_metrics(dekads)


# ------------------------------------------------------------------
# Resumable, one-at-a-time driver
# ------------------------------------------------------------------
out_csv = f"{config.output_dir}/sensitivity_sweep_{RUN_TAG}.csv"

DEFAULTS = {s["param"]: s["default"] for s in SWEEPS.values()}
PLAN = [(name, spec["param"], v)
        for name, spec in SWEEPS.items() for v in spec["values"]]
_ORDER = ("DL_BLEND_ALPHA", "GPD_THRESHOLD_PERCENTILE", "DENSITY_SATURATION_COUNT")


def _key(sweep, value):
    """Stable row key (handles 70 vs 70.0 from a re-read CSV)."""
    return (str(sweep), round(float(value), 4))


def _sig(param, value):
    """Full (alpha, gpd, csat) signature for a swept setting; equal signatures
    share a result, so the three all-default rows are computed only once."""
    trip = dict(DEFAULTS)
    trip[param] = value
    return tuple(round(float(trip[p]), 4) for p in _ORDER)


# Resume from any existing checkpoint.
results, done, by_sig = [], {}, {}
if os.path.isfile(out_csv):
    try:
        prev = pd.read_csv(out_csv)
        results = prev.to_dict("records")
        for r in results:
            met = {v: r.get(v) for v in POOL_VARS}
            if any(not pd.isna(x) for x in met.values()):
                done[_key(r["sweep"], r["value"])] = met
                by_sig[_sig(r["param"], r["value"])] = met
    except Exception as e:  # noqa: BLE001 - corrupt/partial checkpoint, start clean
        print(f"  could not read checkpoint ({e}); starting fresh")
        results, done, by_sig = [], {}, {}

pending = [(n, p, v) for (n, p, v) in PLAN if _key(n, v) not in done]
print(f"{len(done)}/{len(PLAN)} settings done; {len(pending)} pending  ->  {out_csv}")

limit = RUNS_PER_CALL if RUNS_PER_CALL else len(pending)
ran = 0
for (name, param, val) in pending:
    if ran >= limit:
        print(f"\nReached RUNS_PER_CALL={RUNS_PER_CALL}. Re-run this cell for the next setting.")
        break
    print(f"\n===== [{name} = {val}]  (param {param}) =====")
    sig = _sig(param, val)
    if sig in by_sig:
        med = dict(by_sig[sig])
        print(f"    reused all-default result {sig} (no pipeline run)")
    else:
        for s in SWEEPS.values():                # reset all three to defaults,
            set_param(s["param"], s["default"])
        set_param(param, val)                    # then set the swept one
        med = run_setting(DEKADS)
        by_sig[sig] = med
        ran += 1
    row = {"sweep": name, "param": param, "value": val}
    row.update(med)
    results = [r for r in results if _key(r["sweep"], r["value"]) != _key(name, val)]
    results.append(row)
    done[_key(name, val)] = med
    pd.DataFrame(results).to_csv(out_csv, index=False)   # checkpoint every setting
    print("    ->", {k: round(v, 3) for k, v in med.items() if not pd.isna(v)},
          f"(checkpointed {len(results)}/{len(PLAN)} -> {out_csv})")

remaining = [(n, v) for (n, p, v) in PLAN if _key(n, v) not in done]
if not remaining:
    for s in SWEEPS.values():                    # restore operating defaults
        set_param(s["param"], s["default"])
    df = pd.DataFrame(results)
    df.to_csv(out_csv, index=False)
    print(f"\nAll {len(PLAN)} settings complete. Saved: {out_csv}")
    print(df.to_string(index=False))
else:
    print(f"\n{len(remaining)} setting(s) still pending. Re-run to continue.")


## 7 Inspect results

Once every setting is done the last cell prints the full table and `sensitivity_sweep_indonesia.csv` holds the pooled median metric per parameter value. Send the CSV back for the thesis Section 4.8 comparison.